[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrpmichel/Libro_Regularizacion_La_Salle_Bajio/blob/main/int/u5_tecnicas.ipynb)

# Int. U5 · Técnicas de integración
**Libro de regularización en matemáticas para ingeniería · ULSA Bajío**

Este notebook acompaña la unidad 5 del bloque de cálculo integral. Tiene una sección por subtema, en el mismo orden que el libro:

1. Sustitución
2. Integración por partes
3. Fracciones parciales
4. Potencias de funciones trigonométricas y sustitución trigonométrica
5. Integrales impropias

**Cómo usarlo.** Ejecuta las celdas en orden (Entorno de ejecución → Ejecutar todas). Cada sección termina con una celda **Contrasta**: antes de ejecutarla, resuelve a mano el caso que se indica y escribe tu resultado. La referencia de la calculadora aparece solo después de que escribes el tuyo.

**Etiqueta del repositorio.** `INT-U5` (bloque, unidad). Las celdas de código de cada sección vienen de `int/u5_tecnicas/NN_*.py`; los fragmentos que el libro imprime, de `int/u5_tecnicas/fragmentos/`.

**Convenciones.** Punto decimal. Los valores se *redondean* (no se truncan) a las cifras significativas que elijas. Para escribir funciones usa `x` como variable, `*` para multiplicar, `**` o `^` para potencias, `sqrt(...)`, `exp(...)`, `log(...)` (natural), `abs(...)`, `atan(...)` y `asin(...)`; también `pi`, `sen` y `ln`. Los ángulos de `sin`, `cos` y `tan` van en **radianes**. Toda antiderivada se muestra con su constante `+ C`. Las calculadoras de antiderivadas derivan el resultado para comprobarlo y lo comparan con la antiderivada de `sympy`, porque dos respuestas correctas pueden diferir en una constante.

## 0. Versiones e imports

In [ ]:
# Versiones mínimas probadas. En Colab ya vienen instaladas; esta celda solo lo asegura.
%pip install -q "numpy>=1.26" "sympy>=1.12" "matplotlib>=3.8" "ipywidgets>=8" "scipy>=1.11"

import sys, math, re, signal
from decimal import Decimal, ROUND_HALF_UP
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
import ipywidgets as widgets
from IPython.display import display
from scipy.integrate import quad
from scipy.optimize import brentq
from sympy.integrals import manualintegrate as _mi
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor, rationalize)

import scipy
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| scipy", scipy.__version__, "| sympy", sp.__version__,
      "| matplotlib", matplotlib.__version__, "| ipywidgets", widgets.__version__)

In [ ]:
# Símbolos y utilidades compartidas por todo el notebook
x = sp.symbols("x", real=True)
U = sp.symbols("u", real=True)                     # variable de las sustituciones

_FUNCIONES = {"sqrt": sp.sqrt, "abs": sp.Abs, "exp": sp.exp, "log": sp.log, "ln": sp.log,
              "pi": sp.pi, "sin": sp.sin, "sen": sp.sin, "cos": sp.cos, "tan": sp.tan, "sec": sp.sec,
              "atan": sp.atan, "arctan": sp.atan, "asin": sp.asin, "arcsen": sp.asin, "arcsin": sp.asin,
              "acos": sp.acos, "arccos": sp.acos, "sinh": sp.sinh, "cosh": sp.cosh, "E": sp.E, "e": sp.E}
_GLOBAL = {"Integer": sp.Integer, "Float": sp.Float, "Rational": sp.Rational,
           "Symbol": sp.Symbol, "Function": sp.Function}
# rationalize: 0.1 se lee como 1/10, para que los resultados salgan exactos
_TRANSF = standard_transformations + (implicit_multiplication_application, convert_xor, rationalize)
_OTRAS_VARIABLES = ("t", "w", "q", "s")             # se leen como x si son la única variable (u se reserva)
_C = sp.Symbol("C")
_c = sp.Symbol("c")                               # también se acepta + c minúscula


def parsear(texto, variables=("x",)):
    """Convierte un texto en una expresión de sympy o explica qué salió mal.
    Acepta sen y ln, la letra e como número de Euler, π y √, |...| como valor absoluto, sin^2(x),
    una constante + C o + c (se descarta) y otra letra (t, w, q o s) en lugar de x si es la única variable."""
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Escribe una función, por ejemplo: x**2")
    texto = texto.replace("π", "pi").replace("√", "sqrt").replace("·", "*").replace("−", "-")
    texto = re.sub(r"\|([^|]+)\|", r"abs(\1)", texto)                      # |x - 1| -> abs(x - 1)
    texto = re.sub(r"\b(sin|sen|cos|tan)\s*\^\s*(\d+)\s*\(([^()]*)\)", r"(\1(\3))**\2", texto)   # sin^2(x)
    nombres = set(re.findall(r"[A-Za-z_]\w*", texto))
    otras = [v for v in _OTRAS_VARIABLES if v in nombres]
    if "x" not in nombres and len(otras) == 1:
        texto = re.sub(rf"(?<![A-Za-z_]){otras[0]}(?![A-Za-z_0-9])", "x", texto)
        nombres = set(re.findall(r"[A-Za-z_]\w*", texto))
    local = {**_FUNCIONES, "x": x, "C": _C, "c": _c}
    desconocidos = sorted(nombres - set(local))
    if desconocidos:
        raise ValueError(f"No reconozco {desconocidos}. Usa x (o t) como variable, números en lugar de letras para las "
                         "constantes y, si hace falta, sqrt, exp, log o ln, abs, sin o sen, cos, tan, atan, asin o pi.")
    try:
        expr = parse_expr(texto, local_dict=local, global_dict=dict(_GLOBAL), transformations=_TRANSF)
    except Exception as err:
        raise ValueError("No pude leer la función. Usa * para multiplicar, ** o ^ para potencias, punto decimal "
                         "y sqrt(...) para raíces.") from err
    if not isinstance(expr, sp.Expr) or expr.atoms(sp.core.function.AppliedUndef):
        raise ValueError("Escribe una expresión sin el signo = y solo con las funciones permitidas.")
    expr = expr.subs({_C: 0, _c: 0})              # la constante de integración no cambia la derivada
    if expr.has(sp.oo, -sp.oo, sp.zoo, sp.nan):
        raise ValueError("La función no puede contener infinito; los extremos infinitos se escriben aparte.")
    if expr.free_symbols - {x}:
        raise ValueError("Usa una sola variable.")
    return expr


def lista_numeros(texto, nombre="la lista"):
    """'0, 1.5, 3' -> [0.0, 1.5, 3.0], o un mensaje claro si algo no es número."""
    partes = [p for p in re.split(r"[,\s;]+", str(texto).strip()) if p]
    if not partes:
        raise ValueError(f"Escribe {nombre} como números separados por comas, por ejemplo: 0, 1, 3, 4")
    try:
        return [float(p) for p in partes]
    except ValueError as err:
        raise ValueError(f"En {nombre} hay algo que no es un número: usa punto decimal y comas entre valores.") from err


def cifras(valor, n=4):
    """Redondea (no trunca) a n cifras significativas con la regla escolar (5 sube) y conserva los ceros finales.
    Fuera de 1e-4 a 1e7 usa notación científica."""
    v = float(valor)
    if math.isnan(v):
        return "no definido"
    if math.isinf(v):
        return "+∞" if v > 0 else "-∞"
    if v == 0:
        return "0"
    d = Decimal(repr(v))
    if abs(v) >= 1e7 or abs(v) < 1e-4:
        m = (d.scaleb(-d.adjusted())).quantize(Decimal(1).scaleb(-n + 1), rounding=ROUND_HALF_UP)
        e = d.adjusted()
        if abs(m) >= 10:                     # 9.996e5 con 3 cifras pasa a 1.00e6
            m, e = (m / 10).quantize(Decimal(1).scaleb(-n + 1), rounding=ROUND_HALF_UP), e + 1
        return f"{m}e{e}"
    q = d.quantize(Decimal(1).scaleb(d.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() != d.adjusted():          # 9.996 con 3 cifras pasa a 10.0: se reajusta la posición
        q = d.quantize(Decimal(1).scaleb(q.adjusted() - n + 1), rounding=ROUND_HALF_UP)
    if q.adjusted() >= n:                     # enteros grandes: sin notación científica
        return f"{int(q):d}"
    return format(q, "f")


def cerca(p, q, tol=1e-9):
    return abs(float(p) - float(q)) < tol


def coincide(mi, ref, n=3):
    """True si mi y ref coinciden a n cifras significativas (para las celdas Contrasta)."""
    mi, ref = float(mi), float(ref)
    return abs(mi - ref) <= 5 * 10**(-n) * max(abs(ref), 1e-12) or abs(mi - ref) < 1e-12


def a_numero(valor):
    """Número escrito por el alumno: 1.57, 'pi/2' o '3/2'."""
    if isinstance(valor, (int, float)):
        return float(valor)
    v = parsear(str(valor))
    if v.free_symbols:
        raise ValueError("Escribe un número, sin x.")
    return float(sp.N(v))


def _rechaza(fn):
    """True si fn() lanza ValueError (para probar que una calculadora rechaza una entrada)."""
    try:
        fn()
    except ValueError:
        return True
    return False


class _Tarda(Exception):
    pass


def con_limite(fn, segundos=15):
    """Ejecuta fn() con un límite de tiempo (en Colab); si se pasa, lanza ValueError con un mensaje claro."""
    if not hasattr(signal, "SIGALRM"):        # Windows: sin límite
        return fn()
    def _alarma(*_):
        raise _Tarda()
    anterior = signal.signal(signal.SIGALRM, _alarma)
    signal.alarm(segundos)
    try:
        return fn()
    except _Tarda:
        raise ValueError(f"sympy tardó más de {segundos} s con esta función; prueba una más sencilla.") from None
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, anterior)


def valor_real(f, c):
    """f(c) como float, o ValueError si no es un número real finito."""
    v = complex(sp.N(f.subs(x, c)))
    if abs(v.imag) > 1e-12 or not math.isfinite(v.real):
        raise ValueError(f"f no tiene valor real en x = {cifras(float(c), 6)}: revisa el dominio.")
    return v.real


def numerica(f):
    """Versión numpy de f que siempre devuelve un arreglo del tamaño de la entrada."""
    g = sp.lambdify(x, f, "numpy")
    return lambda t: np.asarray(g(np.asarray(t, dtype=float)), dtype=float) * np.ones_like(np.asarray(t, dtype=float))


# Antiderivadas (las mismas herramientas de las Unidades 3 y 4, ampliadas para esta unidad)
_PERMITIDAS = (sp.exp, sp.log, sp.sin, sp.cos, sp.tan, sp.sec, sp.Abs, sp.atan, sp.asin, sp.acos)
_PUNTOS = [-8.9, -5.7, -4.3, -2.9, -2.1, -1.3, -0.7, -0.37, 0.13, 0.37, 0.53, 0.7, 0.91, 1.3, 2.1, 2.9, 4.3, 5.7, 8.9]


def _siempre_positivo(a):
    """True si a > 0 para todo x real (sympy lo sabe, o es un polinomio sin raíces reales y coeficiente principal > 0)."""
    if a.is_positive:
        return True
    if a.has(x) and a.is_polynomial(x):
        P = sp.Poly(a, x)
        return P.count_roots() == 0 and P.LC() > 0
    return False


def con_valor_absoluto(F, conservar=()):
    """ln(u) -> ln|u| salvo que u sea siempre positivo: la antiderivada de u'/u vale donde u > 0 y donde u < 0.
    Los logaritmos de `conservar` (los que ya traía el integrando, como ln x en ∫ x ln x dx) se dejan igual."""
    return F.replace(sp.log, lambda a: sp.log(a) if (_siempre_positivo(a) or isinstance(a, sp.Abs)
                                                     or sp.log(a) in conservar) else sp.log(sp.Abs(a)))


def sin_desfase(F):
    """sen(x + π/4) -> (sen x + cos x)/√2: simplify a veces junta un seno y un coseno en uno solo con desfase."""
    def es_desfase(e):
        return isinstance(e, (sp.sin, sp.cos)) and isinstance(e.args[0], sp.Add) and e.args[0].has(sp.pi)
    if not any(es_desfase(a) for a in F.atoms(sp.Function)):
        return F
    return sp.factor_terms(sp.expand(F.replace(es_desfase, sp.expand_trig)))


def sin_abs_en_log(f):
    """ln|u| -> ln u, solo para integrar (la derivada de las dos es u'/u); el valor absoluto se repone al final."""
    return f.replace(lambda e: isinstance(e, sp.log) and e.args[0].has(sp.Abs),
                     lambda e: sp.log(e.args[0].replace(sp.Abs, lambda a: a)))


def misma_funcion(g, h):
    """True si g y h coinciden: primero con sympy; si no decide, en los puntos donde las dos son reales."""
    try:
        if sp.simplify(g - h) == 0:
            return True
    except Exception:
        pass
    comparados = 0
    for c in _PUNTOS:
        try:
            if abs(valor_real(g, c) - valor_real(h, c)) > 1e-9 * max(1, abs(valor_real(h, c))):
                return False
            comparados += 1
        except (ValueError, TypeError, ZeroDivisionError, OverflowError):
            continue
    return comparados >= 3


def _a_logaritmos(F):
    """asinh, acosh y atanh se escriben con logaritmos (son elementales)."""
    return F.rewrite(sp.log) if F.has(sp.asinh, sp.acosh, sp.atanh) else F


def _rama_real(F, f):
    """De una antiderivada por casos (Piecewise) o con constantes imaginarias, una fórmula real cuya derivada es f."""
    candidatos = [e for e, _ in F.args] if isinstance(F, sp.Piecewise) else [F]
    for e in candidatos:
        for cand in (e, sp.re(e)):
            cand = _a_logaritmos(cand)
            if cand.has(sp.I, sp.re, sp.im, sp.Piecewise):
                continue
            if misma_funcion(derivada_legible(cand), f):
                return cand
    return None


def antiderivada(f):
    """Una antiderivada real de f (con C = 0), simplificada y con ln|..| donde corresponde."""
    g = sin_abs_en_log(f)
    F = con_limite(lambda: sp.integrate(g, x), 20)
    if F.has(sp.Integral):
        raise ValueError("sympy no encontró una antiderivada para esta función.")
    if F.has(sp.RootSum):
        raise ValueError("sympy expresó la antiderivada con raíces de un polinomio; este caso está fuera del curso.")
    F = sp.piecewise_fold(F)
    if F.has(sp.Piecewise, sp.I):
        F = _rama_real(F, g)
        if F is None:
            raise ValueError("sympy dio una antiderivada por casos que no pude reducir a una sola fórmula real.")
    F = _a_logaritmos(F)
    raras = {type(a).__name__ for a in F.atoms(sp.Function) if not isinstance(a, _PERMITIDAS)}
    if raras:
        raise ValueError(f"la antiderivada usa funciones especiales ({', '.join(sorted(raras))}): no es elemental. "
                         "Una integral definida de esta función se calcula con sumas numéricas (Unidad 4).")
    if F.is_polynomial(x):
        F = sp.expand(F)
    else:
        F = con_valor_absoluto(sin_desfase(_a_logaritmos(sp.trigsimp(sp.simplify(F)))), g.atoms(sp.log))
    if not misma_funcion(derivada_legible(F), g):
        raise ValueError("la comprobación F' = f falló; revisa la función.")
    return F


def derivada_legible(F):
    """F' sin el Piecewise que sympy agrega al derivar ln|u| (la derivada de ln|u| es u'/u)."""
    G = F.replace(lambda e: isinstance(e, sp.log) and e.args[0].has(sp.Abs),
                  lambda e: sp.log(e.args[0].replace(sp.Abs, lambda a: a)))
    return sp.simplify(sp.diff(G, x))


def continua_en(f, a, b):
    """True si f es continua en todo el intervalo cerrado entre a y b."""
    lo, hi = sorted((sp.nsimplify(a), sp.nsimplify(b)))
    dom = sp.calculus.util.continuous_domain(f, x, sp.Interval(lo, hi))
    return dom == sp.Interval(lo, hi)


# Qué regla eligió sympy: nombres en español de las reglas de integral_steps
_NOMBRES = {
    "URule": "sustitución", "PartsRule": "integración por partes", "CyclicPartsRule": "partes (cíclica)",
    "RewriteRule": "reescritura del integrando (identidad, división o fracciones parciales)",
    "TrigSubstitutionRule": "sustitución trigonométrica",
    "SqrtQuadraticRule": "raíz de una cuadrática (sustitución trigonométrica)",
    "SqrtQuadraticDenomRule": "raíz de una cuadrática (sustitución trigonométrica)",
    "ReciprocalSqrtQuadraticRule": "raíz de una cuadrática (sustitución trigonométrica)",
    "CompleteSquareRule": "completar el cuadrado", "ArctanRule": "tabla (arco tangente)",
    "ArcsinRule": "tabla (arco seno)",
    "DontKnowRule": "integral_steps no reconoce la regla (sympy.integrate puede resolverla de otra forma)",
}
_TABLA = {"PowerRule", "ExpRule", "SinRule", "CosRule", "ReciprocalRule", "ConstantRule", "Sec2Rule"}
_ENVOLTURAS = {"AlternativeRule", "ConstantTimesRule"}


def _desenvuelve(r):
    while type(r).__name__ in _ENVOLTURAS:
        r = r.alternatives[0] if type(r).__name__ == "AlternativeRule" else r.substep
    return r


def regla_sympy(f):
    """Nombre en español de la técnica principal que integral_steps usa para f (la primera que no es envoltura)."""
    try:
        r = _desenvuelve(con_limite(lambda: _mi.integral_steps(f, x), 8))
    except ValueError:
        return "integral_steps tardó demasiado; se omite"
    nombre = type(r).__name__
    if nombre == "AddRule":
        partes = []
        for s_ in r.substeps:
            n_ = type(_desenvuelve(s_)).__name__
            partes.append("tabla de la Unidad 3" if n_ in _TABLA else _NOMBRES.get(n_, n_))
        return "suma término por término: " + ", ".join(dict.fromkeys(partes))
    if nombre in _TABLA:
        return "tabla de la Unidad 3"
    return _NOMBRES.get(nombre, nombre)


def _constante(dif):
    """Valor de una diferencia que debería ser constante: exacto si se puede, o el primero real de _PUNTOS."""
    dif = sp.simplify(dif)
    if not dif.has(x):
        v = dif
    else:
        v = None
        for c in _PUNTOS:
            try:
                v = valor_real(dif, c); break
            except (ValueError, TypeError, ZeroDivisionError, OverflowError):
                continue
        if v is None:
            return None
    vf = float(sp.N(v))
    if abs(vf) < 1e-10:
        return sp.Integer(0)
    return v if not isinstance(v, float) else sp.nsimplify(round(vf, 10), [sp.pi, sp.E], tolerance=1e-9)


def compara_con_sympy(F, f):
    """(es_correcta, constante): F' = f, y en cuánto difiere F de la antiderivada de sympy (None si no se sabe)."""
    if not misma_funcion(derivada_legible(F), sin_abs_en_log(f)):
        return False, None
    try:
        G = antiderivada(f)
    except ValueError:
        return True, None
    return True, _constante(F - G)


def mostrar_antiderivada(F):
    """F + C, con la constante siempre a la vista."""
    return f"{F} + C"


def texto_constante(cte):
    if cte is None:
        return ""
    return "coincide con la antiderivada de sympy" if cte == 0 else f"difiere de la antiderivada de sympy en la constante {cte}"

## 1. Sustitución

$$\int f\big(g(x)\big)\,g'(x)\,dx=\int f(u)\,du=F\big(g(x)\big)+C,\qquad u=g(x),\quad du=g'(x)\,dx.$$

En una integral definida, los límites cambian con la variable: $\int_a^b f\big(g(x)\big)g'(x)\,dx=\int_{g(a)}^{g(b)}f(u)\,du$.

Primero, qué técnica elige `sympy` para cada integral de la semilla. La función `regla_sympy` lee la estructura de pasos de `integral_steps`.

In [ ]:
semilla = [2 * x * sp.cos(x**2), x * sp.cos(x), 1 / (x**2 - 1), sp.sin(x)**2]
for f in semilla:
    print(f"∫ {f} dx  →  {regla_sympy(f)}")
print("∫_1^∞ dx/x² =", sp.integrate(1 / x**2, (x, 1, sp.oo)), " (impropia, sección 5)")

Un ejemplo con cambio de límites: $\int_0^1\frac{x}{1+x^2}\,dx$ con $u=1+x^2$, que va de $1$ a $2$.

In [ ]:
directa = sp.integrate(x / (1 + x**2), (x, 0, 1))
en_u = sp.Rational(1, 2) * sp.integrate(1 / U, (U, 1, 2))
print("en x:", directa, "  en u:", en_u, "  ≈", cifras(directa, 4))

**Calculadora de sustitución.** Escribe $f(x)$ y tu propuesta de $u(x)$. La calculadora calcula $du$ e intenta escribir $f(x)\,dx$ como $h(u)\,du$. Si queda una $x$, intenta despejarla de $u=g(x)$ y te avisa si no lo logra. Si la sustitución sirve, integra en $u$, regresa a $x$ con la constante $C$, deriva el resultado y lo compara con la antiderivada de `sympy`.

In [ ]:
def _nunca_negativa(g):
    """True si g(x) >= 0 en todos los puntos de prueba donde es real (como x² + 1 o √x)."""
    if g.is_nonnegative:
        return True
    valores = []
    for c in _PUNTOS:
        try:
            valores.append(valor_real(g, c))
        except (ValueError, TypeError, ZeroDivisionError, OverflowError):
            continue
    return len(valores) >= 3 and min(valores) >= 0


def _integra_en_u(hu):
    """∫ h(u) du con las reglas "a mano" de sympy (manualintegrate): rápidas y sin funciones especiales."""
    H = con_limite(lambda: _mi.manualintegrate(sp.expand(hu), U), 8)   # expandido: u^(5/2) en vez de (u - 1)² √u
    if H.has(sp.Integral):
        raise ValueError(f"la integral en u, ∫ {hu} du, no está en la tabla: esa sustitución no la simplifica.")
    H = _a_logaritmos(H)
    raras = [a for a in H.atoms(sp.Function) if not isinstance(a, _PERMITIDAS)]
    if H.has(sp.Piecewise) or raras:           # caso raro: se usa la antiderivada general, escrita en u
        H = antiderivada(hu.subs(U, x)).subs(x, U)
    return H


def sustituir(f, g):
    """∫ f(x) dx con u = g(x). Devuelve du/dx, h(u) con f dx = h du, H(u), F(x) y la lista de despejes usados.
    Si al despejar x hay varias ramas (x = ±√u), se queda con la que cumple F' = f.
    ValueError si queda una x suelta, si la integral en u no está en la tabla o si ninguna rama sirve."""
    if not g.has(x):
        raise ValueError("u debe depender de x.")
    if sp.simplify(g - x) == 0:
        raise ValueError("u = x no cambia la integral: elige como u una parte del integrando "
                         "(lo de adentro de una raíz, de un coseno o de una exponencial).")
    dg = sp.diff(g, x)
    if sp.simplify(dg) == 0:
        raise ValueError("u es constante: su derivada es cero.")
    h = sp.simplify(f / dg)                    # f dx = (f / g') du
    candidatos = []
    hu = sp.simplify(h.subs(g, U))
    if not hu.has(x):
        candidatos.append((hu, None))
    else:                                      # queda una x: se intenta despejar de u = g(x)
        try:
            inversas = con_limite(lambda: sp.solve(sp.Eq(U, g), x), 8)
        except (ValueError, NotImplementedError):
            inversas = []
        for inv in inversas:
            if inv.has(sp.I):                  # raíces complejas de u = x³: no sirven en los reales
                continue
            cand = sp.simplify(h.subs(x, inv))
            if cand.has(x):
                continue
            previo = [k for k, (hc, _) in enumerate(candidatos) if sp.simplify(hc - cand) == 0]
            if previo:                         # otra rama da el mismo h(u): x solo aparece al cuadrado
                candidatos[previo[0]][1].append(inv)
            else:
                candidatos.append((cand, [inv]))
    if not candidatos:
        raise ValueError("queda una x que no se puede escribir con u: esa sustitución no sirve.")
    positiva, error = _nunca_negativa(g), None
    for hu, despeje in candidatos:
        if positiva:                           # u >= 0: |u| = u
            hu = hu.subs(sp.Abs(U), U)
        try:
            H = _integra_en_u(hu)
        except ValueError as err:
            error = err; continue
        F = con_valor_absoluto(sp.simplify(H.subs(U, g)))
        if misma_funcion(derivada_legible(F), sin_abs_en_log(f)):
            return {"du": dg, "hu": hu, "H": H, "F": F, "despeje": despeje}
    if error is not None and len(candidatos) == 1:
        raise error
    raise ValueError("al despejar x de u = g(x) hay que elegir un signo (x = ±...) y ninguna elección sirve para "
                     "todo el dominio: esa sustitución no funciona aquí.")


def calculadora_sustitucion(f_txt, u_txt):
    try:
        f, g = parsear(f_txt), parsear(u_txt)
        r = sustituir(f, g)
    except ValueError as err:
        print("No sirve o hay un error en la entrada:", err); return
    print(f"u = {g}     du = ({r['du']}) dx")
    if r["despeje"]:
        ramas = " o x = ".join(str(d) for d in r["despeje"])
        print(f"quedaba una x; de u = {g} se despejó x = {ramas}"
              + ("  (las dos ramas dan el mismo integrando en u)" if len(r["despeje"]) > 1 else ""))
    print(f"∫ f(x) dx = ∫ {r['hu']} du = {r['H']} + C")
    print(f"regreso a x: {mostrar_antiderivada(r['F'])}")
    ok, cte = compara_con_sympy(r["F"], f)
    print("derivada de tu resultado = f(x):", "sí" if ok else "NO")
    if texto_constante(cte):
        print(texto_constante(cte))
    print("regla que eligió sympy:", regla_sympy(f))


widgets.interact(calculadora_sustitucion,
    f_txt=widgets.Text(value="x*cos(x^2)", description="f(x) =", continuous_update=False),
    u_txt=widgets.Text(value="x^2", description="u(x) =", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 1 (resultado conocido)
PRUEBAS_1 = [
    ("x cos x² con u = x²: ½ sen x²", lambda: cerca(valor_real(sustituir(x * sp.cos(x**2), x**2)["F"] - sp.sin(x**2) / 2, 0.7), 0)),
    ("x√(x+1) con u = x+1: despeja x = u - 1",
     lambda: (lambda r: bool(r["despeje"]) and compara_con_sympy(r["F"], x * sp.sqrt(x + 1))[0])(
         sustituir(x * sp.sqrt(x + 1), x + 1))),
    ("cos x³ con u = x³: no sirve (integral en u fuera de la tabla)", lambda: _rechaza(lambda: sustituir(sp.cos(x**3), x**3))),
    ("x cos x³ con u = x³: no sirve", lambda: _rechaza(lambda: sustituir(x * sp.cos(x**3), x**3))),
    ("tan x con u = cos x: -ln|cos x|",
     lambda: compara_con_sympy(sustituir(sp.tan(x), sp.cos(x))["F"], sp.tan(x))[0]
     and sustituir(sp.tan(x), sp.cos(x))["F"].has(sp.Abs)),
    ("2x² con u = x²: no sirve (x = ±√u, ninguna rama vale en todo el dominio)",
     lambda: _rechaza(lambda: sustituir(2 * x**2, x**2))),
    ("u = x se rechaza", lambda: _rechaza(lambda: sustituir(x * sp.cos(x), x))),
]
for nombre, prueba in PRUEBAS_1:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 1).** Calcula a mano $\int x^2e^{x^3}\,dx$ y escribe tu antiderivada como texto, sin la $C$ (por ejemplo `"x**2/2"`).

In [ ]:
mi_respuesta = None     # escribe tu antiderivada entre comillas

if mi_respuesta is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    f = x**2 * sp.exp(x**3)
    try:
        ok, cte = compara_con_sympy(parsear(mi_respuesta), f)
    except ValueError as err:
        print("Revisa tu respuesta:", err)
    else:
        print("La calculadora da:", mostrar_antiderivada(sustituir(f, x**3)["F"]))
        print("tu respuesta es correcta" + (f" (difiere en la constante {cte})" if cte not in (None, 0) else "")
              if ok else "NO coincide: con u = x³, du = 3x² dx, así que x² dx = du/3")

## 2. Integración por partes

$$\int u\,dv=uv-\int v\,du,\qquad\int_a^b u\,v'\,dx=\Big[uv\Big]_a^b-\int_a^b v\,u'\,dx.$$

Sale de la regla del producto, $(uv)'=u'v+uv'$. La figura del libro lo muestra con áreas: para $y=\ln x$ en $[1,e]$, el área bajo la curva más el área a su izquierda llenan el rectángulo $e\times1$.

In [ ]:
bajo = sp.integrate(sp.log(x), (x, 1, sp.E))
izquierda = sp.integrate(sp.exp(U), (U, 0, 1))
print("bajo ln x:", bajo, "  a la izquierda:", izquierda, "  suma:", sp.simplify(bajo + izquierda), "= e")

**Calculadora de partes.** Escribe $f(x)$ y tu elección de $u$; el resto es $dv=\frac{f}{u}\,dx$. La calculadora muestra $du$, $v$ y la fórmula $\int f\,dx=uv-\int v\,du$ con la integral nueva, y te avisa si esa integral es más complicada que la original: si la contiene, si sube el grado del polinomio, si aparece una función trascendente que no estaba o, con lo demás igual, si tiene más operaciones. Si la integral nueva es un polinomio por una exponencial o una función trigonométrica, aplica partes otra vez, hasta tres veces.

In [ ]:
_CLASE = {sp.exp: "exp", sp.log: "ln", sp.sin: "trig", sp.cos: "trig", sp.tan: "trig", sp.sec: "trig",
          sp.atan: "arco", sp.asin: "arco", sp.acos: "arco"}


def _tipos(g):
    """Clases de funciones trascendentes en g: exp, ln, trig (sen, cos y tan cuentan igual) y arco."""
    return {_CLASE.get(type(a), type(a).__name__) for a in g.atoms(sp.Function)}


def _grado_polinomial(g):
    """Grado del factor polinomial de g (0 si no tiene)."""
    poli = sp.Mul(*[a for a in sp.Mul.make_args(sp.factor(g)) if a.has(x) and a.is_polynomial(x)])
    return sp.degree(poli, x) if poli.has(x) else 0


def _empeora(nueva, f):
    """La integral nueva es peor si contiene a la original, si aparece una función trascendente que no estaba,
    si sube el grado del polinomio
    o, con las mismas funciones y el mismo grado, si tiene más operaciones. Si desaparece una función
    trascendente (ln x o arctan x se derivan y se van), la integral nueva es mejor."""
    if any(not sp.simplify(t / f).has(x) for t in sp.Add.make_args(sp.expand(nueva))):
        return True                            # la integral nueva contiene a la original: no se avanzó
    tn, tf = _tipos(nueva), _tipos(f)
    if tn < tf:
        return False
    if not tn <= tf:
        return True
    gn, gf = _grado_polinomial(nueva), _grado_polinomial(f)
    return gn > gf or (gn == gf and sp.count_ops(nueva) > sp.count_ops(f) + 1)


def partes(f, u_expr):
    """Un paso: u = u_expr, dv = f/u. Devuelve u, du, dv, v, la integral nueva v du y si la elección empeoró la integral."""
    if not u_expr.has(x):
        raise ValueError("u debe depender de x.")
    dv = sp.simplify(f / u_expr)
    try:
        v = antiderivada(dv)
    except ValueError:
        raise ValueError(f"con u = {u_expr} queda dv = ({dv}) dx, y ese dv no se sabe integrar: elige otra u.") from None
    du = sp.diff(u_expr, x)
    nueva = sp.simplify(v * du)
    return {"f": f, "u": u_expr, "du": du, "dv": dv, "v": v, "nueva": nueva, "empeora": _empeora(nueva, f)}


def _parte_polinomial(g):
    """Si g es un polinomio de grado >= 1 por una exponencial o una función trigonométrica, el polinomio
    (la siguiente u); si no, None."""
    fac = sp.Mul.make_args(sp.factor(g))
    poli = sp.Mul(*[a for a in fac if a.has(x) and a.is_polynomial(x)])
    resto = sp.Mul(*[a for a in fac if not (a.has(x) and a.is_polynomial(x))])
    sin_cocientes = not any(isinstance(a, sp.Pow) and a.exp.is_negative for a in sp.Mul.make_args(resto))
    if poli.has(x) and _tipos(resto) and _tipos(resto) <= {"exp", "trig"} and sin_cocientes:
        return poli
    return None


def por_partes(f, u_expr, max_pasos=3):
    """Aplica partes hasta max_pasos veces. Devuelve la lista de pasos y la antiderivada F (con C = 0).
    El primer paso usa la u del alumno; los siguientes, el factor polinomial de la integral nueva."""
    pasos, signo, acumulado, actual, u_act = [], 1, 0, f, u_expr
    for k in range(max_pasos):
        try:
            p = partes(actual, u_act)
        except ValueError:
            if k == 0:
                raise
            break                              # el paso siguiente no se puede: se integra lo que queda
        pasos.append(p)
        acumulado += signo * p["u"] * p["v"]
        signo, actual = -signo, p["nueva"]     # F = uv - ∫ v du: la integral nueva entra con el signo cambiado
        if p["empeora"]:
            break
        u_act = _parte_polinomial(actual)
        if u_act is None:
            break
    try:
        resto = antiderivada(actual)
    except ValueError:
        raise ValueError(f"la integral que queda, ∫ {actual} dx, no tiene una antiderivada elemental; "
                         "prueba otra elección de u.") from None
    return pasos, sin_desfase(sp.simplify(acumulado + signo * resto))


def calculadora_partes(f_txt, u_txt):
    try:
        f, u_expr = parsear(f_txt), parsear(u_txt)
        pasos, F = por_partes(f, u_expr)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    for k, p in enumerate(pasos, 1):
        print(f"paso {k}: u = {p['u']}, dv = ({p['dv']}) dx  →  du = ({p['du']}) dx, v = {p['v']}")
        print(f"        ∫ {p['f']} dx = {p['u'] * p['v']} - ∫ {p['nueva']} dx")
        if p["empeora"]:
            print("        AVISO: la integral nueva es más complicada que la original; prueba otra elección de u")
    print("resultado:", mostrar_antiderivada(F))
    ok, cte = compara_con_sympy(F, f)
    print("derivada del resultado = f(x):", "sí" if ok else "NO")
    if texto_constante(cte):
        print(texto_constante(cte))
    print("regla que eligió sympy:", regla_sympy(f))


widgets.interact(calculadora_partes,
    f_txt=widgets.Text(value="x*exp(x)", description="f(x) =", continuous_update=False),
    u_txt=widgets.Text(value="x", description="u =", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 2 (resultado conocido)
PRUEBAS_2 = [
    ("x e^x con u = x: (x - 1)e^x", lambda: sp.simplify(por_partes(x * sp.exp(x), x)[1] - (x - 1) * sp.exp(x)) == 0),
    ("x e^x con u = e^x: la integral nueva empeora", lambda: partes(x * sp.exp(x), sp.exp(x))["empeora"]),
    ("ln x con u = ln x: x ln x - x", lambda: misma_funcion(por_partes(sp.log(x), sp.log(x))[1], x * sp.log(x) - x)),
    ("x² sen x con u = x²: dos pasos, -x² cos x + 2x sen x + 2 cos x",
     lambda: (lambda r: len(r[0]) == 2 and sp.simplify(r[1] - (-x**2 * sp.cos(x) + 2 * x * sp.sin(x) + 2 * sp.cos(x))) == 0)(
         por_partes(x**2 * sp.sin(x), x**2))),
    ("x ln x con u = ln x: la integral nueva (x/2) no empeora", lambda: not partes(x * sp.log(x), sp.log(x))["empeora"]),
    ("x e^x con u = x²: dv = e^x/x no se integra", lambda: _rechaza(lambda: partes(x * sp.exp(x), x**2))),
]
for nombre, prueba in PRUEBAS_2:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 2).** Calcula a mano $\int x\cos x\,dx$ y escribe tu antiderivada como texto, sin la $C$.

In [ ]:
mi_respuesta = None     # escribe tu antiderivada entre comillas

if mi_respuesta is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    f = x * sp.cos(x)
    try:
        ok, cte = compara_con_sympy(parsear(mi_respuesta), f)
    except ValueError as err:
        print("Revisa tu respuesta:", err)
    else:
        print("La calculadora da:", mostrar_antiderivada(por_partes(f, x)[1]))
        print("tu respuesta es correcta" + (f" (difiere en la constante {cte})" if cte not in (None, 0) else "")
              if ok else "NO coincide: con u = x y dv = cos x dx, v = sen x; revisa el signo de ∫ sen x dx")

## 3. Fracciones parciales

Una función racional propia $\frac{P}{Q}$ se escribe como suma de fracciones simples: $\frac{A}{x-r}$ por cada factor lineal, $\frac{A_1}{x-r}+\dots+\frac{A_k}{(x-r)^k}$ por cada factor lineal repetido y $\frac{Bx+C}{x^2+bx+c}$ por cada factor cuadrático irreducible. Si no es propia, primero se divide.

`sympy.apart` hace la descomposición. Aquí se comprueba con el ejemplo del libro, $\frac{2}{x^2-1}$.

In [ ]:
f = 2 / (x**2 - 1)
piezas = sp.apart(f, x)
print("piezas:", piezas, "   suman f:", sp.simplify(piezas - f) == 0)
print("∫_2^3 =", sp.simplify(sp.integrate(f, (x, 2, 3))), "≈", cifras(sp.integrate(f, (x, 2, 3)), 6))

**Calculadora de fracciones parciales.** Escribe una función racional. La calculadora revisa los grados (y divide si hace falta), factoriza el denominador, muestra la descomposición, comprueba que las piezas suman la original e integra término por término. Si hay un factor cuadrático irreducible, te muestra cómo completar el cuadrado. Los denominadores con factores de grado mayor que 2 que no se factorizan con coeficientes racionales quedan fuera del curso y la calculadora los rechaza.

In [ ]:
def _integra_pieza(t):
    """∫ c·g dx = c ∫ g dx: la constante se separa, para que salga ¼ ln|x - 2| y no ¼ ln|4x - 8|."""
    c, g = t.as_independent(x, as_Add=False)
    return c * antiderivada(g)


def fracciones(f):
    """Descomposición e integral de una función racional. ValueError si f no es un cociente de polinomios
    o si el denominador tiene un factor de grado mayor que 2 sin factorizar."""
    num, den = sp.fraction(sp.cancel(sp.together(f)))
    if not (num.is_polynomial(x) and den.is_polynomial(x)):
        raise ValueError("f debe ser un cociente de polinomios en x.")
    if not den.has(x):
        raise ValueError(f"al simplificar, f = {sp.expand(num / den)} es un polinomio (no queda x en el denominador): "
                         "intégralo con la tabla de la Unidad 3.")
    constante, factores = sp.factor_list(den)
    grandes = [g for g, _ in factores if sp.degree(g, x) > 2]
    if grandes:
        raise ValueError(f"el factor {grandes[0]} del denominador tiene grado {sp.degree(grandes[0], x)} y no se "
                         "factoriza con coeficientes racionales: ese caso está fuera del curso.")
    cociente, resto = sp.div(sp.Poly(num, x), sp.Poly(den, x))
    propia = sp.degree(num, x) < sp.degree(den, x)
    cuadraticos = [g for g, _ in factores if sp.degree(g, x) == 2 and sp.discriminant(g, x) < 0]
    irracionales = [g for g, _ in factores if sp.degree(g, x) == 2 and sp.discriminant(g, x) > 0]
    piezas = sp.apart(resto.as_expr() / den, x)
    terminos = [t for t in [cociente.as_expr(), *sp.Add.make_args(piezas)] if t != 0]
    F = sp.Add(*[_integra_pieza(t) for t in terminos])
    return {"propia": propia, "cociente": cociente.as_expr(), "constante": constante, "factores": factores,
            "cuadraticos": cuadraticos,
            "irracionales": irracionales, "piezas": piezas, "F": F,
            "suman": sp.simplify(cociente.as_expr() + piezas - f) == 0}


def completar_cuadrado(g):
    """a x² + b x + c -> a·[(x + h)² + k] con h = b/(2a) y k = c/a - h², escrito para leerse."""
    a, b, c = sp.Poly(g, x).all_coeffs()
    h, k = sp.nsimplify(b / (2 * a)), sp.nsimplify(c / a - (b / (2 * a))**2)
    cuadrado = "x²" if h == 0 else f"(x {'+' if h > 0 else '-'} {abs(h)})²"
    texto = f"{cuadrado} {'+' if k >= 0 else '-'} {abs(k)}"
    return texto if a == 1 else f"{a}·[{texto}]"


def calculadora_fracciones(f_txt):
    try:
        f = parsear(f_txt)
        r = fracciones(f)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    if not r["propia"]:
        print(f"no es propia: la división da {r['cociente']} más una fracción propia")
    piezas_den = [f"({g})^{k}" if k > 1 else f"({g})" for g, k in r["factores"]]
    print("denominador factorizado:", " · ".join(([str(r["constante"])] if r["constante"] != 1 else []) + piezas_den))
    for g in r["cuadraticos"]:
        print(f"factor cuadrático irreducible {g} = {completar_cuadrado(g)}  →  da un logaritmo y un arco tangente")
    for g in r["irracionales"]:
        raices = ", ".join(str(v) for v in sp.solve(g, x))
        print(f"AVISO: {g} tiene raíces reales irracionales ({raices}); con coeficientes racionales la descomposición "
              "lo deja junto, y su integral da logaritmos")
    print("descomposición:", r["piezas"] if r["cociente"] == 0 else f"{r['cociente']} + ({r['piezas']})")
    print("las piezas suman la función original:", "sí" if r["suman"] else "NO")
    print("∫ f dx =", mostrar_antiderivada(r["F"]))
    ok, cte = compara_con_sympy(r["F"], f)
    print("derivada del resultado = f(x):", "sí" if ok else "NO")
    if texto_constante(cte):
        print(texto_constante(cte))
    print("regla que eligió sympy:", regla_sympy(f))


widgets.interact(calculadora_fracciones,
    f_txt=widgets.Text(value="(x+5)/(x^2+x-2)", description="f(x) =", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 3 (resultado conocido)
PRUEBAS_3 = [
    ("(x+5)/(x²+x-2) = 2/(x-1) - 1/(x+2)",
     lambda: sp.simplify(fracciones((x + 5) / (x**2 + x - 2))["piezas"] - (2 / (x - 1) - 1 / (x + 2))) == 0),
    ("1/(x(x-1)²) = 1/x - 1/(x-1) + 1/(x-1)²",
     lambda: sp.simplify(fracciones(1 / (x * (x - 1)**2))["piezas"] - (1 / x - 1 / (x - 1) + 1 / (x - 1)**2)) == 0),
    ("(2x+3)/(x²+2x+5): factor irreducible y F' = f",
     lambda: (lambda r: len(r["cuadraticos"]) == 1 and compara_con_sympy(r["F"], (2 * x + 3) / (x**2 + 2 * x + 5))[0])(
         fracciones((2 * x + 3) / (x**2 + 2 * x + 5)))),
    ("x³/(x²-1): no es propia, cociente x", lambda: (lambda r: not r["propia"] and r["cociente"] == x)(fracciones(x**3 / (x**2 - 1)))),
    ("sen x se rechaza (no es racional)", lambda: _rechaza(lambda: fracciones(sp.sin(x)))),
    ("1/(x³ - 2) se rechaza (factor de grado 3)", lambda: _rechaza(lambda: fracciones(1 / (x**3 - 2)))),
    ("1/(x² - 4): ¼ ln|x - 2| - ¼ ln|x + 2|, sin constantes sueltas",
     lambda: sp.simplify(fracciones(1 / (x**2 - 4))["F"]
                         - (sp.log(sp.Abs(x - 2)) / 4 - sp.log(sp.Abs(x + 2)) / 4)) == 0),
]
for nombre, prueba in PRUEBAS_3:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 3).** Calcula a mano $\int\frac{dx}{x^2-4}$ y escribe tu antiderivada como texto, sin la $C$.

In [ ]:
mi_respuesta = None     # escribe tu antiderivada entre comillas

if mi_respuesta is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    f = 1 / (x**2 - 4)
    try:
        ok, cte = compara_con_sympy(parsear(mi_respuesta), f)
    except ValueError as err:
        print("Revisa tu respuesta:", err)
    else:
        print("La calculadora da:", mostrar_antiderivada(fracciones(f)["F"]))
        print("tu respuesta es correcta" + (f" (difiere en la constante {cte})" if cte not in (None, 0) else "")
              if ok else "NO coincide: 1/(x² - 4) = (1/4)/(x - 2) - (1/4)/(x + 2)")

## 4. Potencias trigonométricas y sustitución trigonométrica

Identidades de reducción: $\;\sin^2x=\frac{1-\cos2x}{2},\quad\cos^2x=\frac{1+\cos2x}{2},\quad\sin x\cos x=\frac12\sin2x.$

Para $\int\sin^mx\cos^nx\,dx$: si $m$ es impar se usa $u=\cos x$; si $n$ es impar, $u=\sin x$; si los dos son pares, se bajan las potencias.

Sustitución trigonométrica ($a>0$): $\sqrt{a^2-x^2}$ con $x=a\sin\theta$; $\sqrt{a^2+x^2}$ con $x=a\tan\theta$; $\sqrt{x^2-a^2}$ con $x=a/\cos\theta$.

In [ ]:
print("∫_0^{2π} sen² x dx =", sp.integrate(sp.sin(x)**2, (x, 0, 2 * sp.pi)), " (la mitad del rectángulo 2π × 1)")
a_ = sp.symbols("a", positive=True)
print("∫_0^a √(a² - x²) dx =", sp.integrate(sp.sqrt(a_**2 - x**2), (x, 0, a_)), " (cuarto de círculo)")

**Calculadora de potencias.** Escribe los exponentes $m$ y $n$ de $\sin^mx\cos^nx$. Te dice qué caso aplica, muestra el integrando reescrito y el resultado.

In [ ]:
from sympy.simplify.fu import TR8 as _TR8      # productos y potencias de seno y coseno -> sumas


def potencias_trig(m, n):
    """(caso, integrando reescrito, F) para ∫ sen^m x cos^n x dx, con m y n enteros no negativos.
    En los casos (a) y (b) el integrando reescrito está en u; en el (c), en x, ya sin potencias (TR8)."""
    if int(m) != m or int(n) != n or m < 0 or n < 0:
        raise ValueError("m y n deben ser enteros no negativos.")
    m, n = int(m), int(n)
    if m % 2 == 1:
        caso = "(a) m impar: separa un sen x, cambia sen² = 1 - cos² y usa u = cos x (du = -sen x dx)"
        reescrita = sp.expand(-(1 - U**2)**((m - 1) // 2) * U**n)
        F = sp.integrate(reescrita, U).subs(U, sp.cos(x))
    elif n % 2 == 1:
        caso = "(b) n impar: separa un cos x, cambia cos² = 1 - sen² y usa u = sen x (du = cos x dx)"
        reescrita = sp.expand(U**m * (1 - U**2)**((n - 1) // 2))
        F = sp.integrate(reescrita, U).subs(U, sp.sin(x))
    else:
        caso = "(c) los dos pares: baja las potencias con las identidades de reducción"
        reescrita = sp.sin(x)**m * sp.cos(x)**n
        for _ in range(8):                     # TR8 repetido hasta que solo queden cosenos de múltiplos de x
            nueva = sp.expand(_TR8(sp.expand(reescrita)))
            if nueva == reescrita:
                break
            reescrita = nueva
        F = sp.integrate(reescrita, x)
    return caso, reescrita, sp.expand(F)


def calculadora_potencias(m, n):
    try:
        caso, reescrita, F = potencias_trig(m, n)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    f = sp.sin(x)**m * sp.cos(x)**n
    print("∫", f, "dx")
    print("caso:", caso)
    print("integrando reescrito:", reescrita, "(en x)" if m % 2 == 0 and n % 2 == 0 else "(en u, con du ya incluido)")
    print("resultado:", mostrar_antiderivada(F))
    print("derivada del resultado = integrando:", "sí" if compara_con_sympy(F, f)[0] else "NO")


widgets.interact(calculadora_potencias,
    m=widgets.IntSlider(value=3, min=0, max=6, description="m (sen)", continuous_update=False),
    n=widgets.IntSlider(value=0, min=0, max=6, description="n (cos)", continuous_update=False));

**Calculadora de sustitución trigonométrica.** Escribe una función con una raíz $\sqrt{a^2-x^2}$, $\sqrt{a^2+x^2}$ o $\sqrt{x^2-a^2}$ (también sirven potencias como $(x^2+9)^{3/2}$). La calculadora reconoce el caso, propone la sustitución, dibuja el triángulo, integra en $\theta$ y regresa a $x$ con las razones del triángulo. En el caso $\sqrt{x^2-a^2}$ trabaja con $x>a$.

In [ ]:
TH = sp.symbols("theta", real=True)
_S, _K = sp.sin(TH), sp.cos(TH)


def _raiz_cuadratica(f):
    """La potencia (±x² + c)^(k/2) de f, con su signo de x² y su constante c; ValueError si hay dos raíces distintas."""
    bases = []
    for p in f.atoms(sp.Pow):
        if p.exp.is_Rational and p.exp.q == 2 and p.base.is_polynomial(x):
            b = sp.expand(p.base)
            if b not in bases:
                bases.append(b)
    if len(bases) > 1:
        raise ValueError("f tiene dos raíces distintas; la sustitución trigonométrica trabaja con una sola.")
    for b in bases:
        coef = sp.Poly(b, x).all_coeffs()
        if len(coef) == 3 and coef[1] == 0 and coef[0] in (1, -1) and coef[2] != 0:
            return b, coef[0], coef[2]
    return None


def _razones(caso, a):
    """Triángulo de referencia: sen θ, cos θ y tan θ escritos con x, y θ como función de x."""
    if caso == "a² - x²":       # x = a sen θ: opuesto x, hipotenusa a, adyacente √(a² - x²)
        return {_S: x / a, _K: sp.sqrt(a**2 - x**2) / a, sp.tan(TH): x / sp.sqrt(a**2 - x**2)}, sp.asin(x / a)
    if caso == "a² + x²":       # x = a tan θ: opuesto x, adyacente a, hipotenusa √(a² + x²)
        return {sp.tan(TH): x / a, _K: a / sp.sqrt(a**2 + x**2), _S: x / sp.sqrt(a**2 + x**2)}, sp.atan(x / a)
    # x = a/cos θ (x > a): hipotenusa x, adyacente a, opuesto √(x² - a²)
    return {_K: a / x, _S: sp.sqrt(x**2 - a**2) / x, sp.tan(TH): sp.sqrt(x**2 - a**2) / a}, sp.acos(a / x)


def _forma_estandar(H):
    """Reescribe H(θ) solo con sen θ, cos θ y θ. sympy da ∫ sec θ dθ como ½ln(sen θ + 1) - ½ln(sen θ - 1);
    como cos²θ = (1 - sen θ)(1 + sen θ), eso es ln|sec θ + tan θ|, y con el triángulo queda real.
    Igual con ∫ csc θ dθ = ln|csc θ - cot θ|."""
    H = H.replace(lambda e: isinstance(e, sp.tan) and e.args[0] == TH / 2, lambda e: _S / (1 + _K))
    H = sp.expand_trig(sp.expand(H))
    for menos, mas, otra in ((_S, 1 + _S, _K), (_K, 1 + _K, _S)):
        H = H.replace(lambda e: isinstance(e, sp.log) and sp.expand(e.args[0] ** 2 - (menos - 1) ** 2) == 0,
                      lambda e: 2 * sp.log(otra) - sp.log(mas))     # |sen θ - 1| = cos²θ/(1 + sen θ)
    return sp.expand(H)


def _sin_constantes(F):
    """Quita los sumandos que no dependen de x (como -ln 3): F sigue siendo antiderivada."""
    F = sp.expand(sp.expand_log(F, force=True))
    return sp.Add(*[t for t in sp.Add.make_args(F) if t.has(x)])


def _ordena(F):
    """Simplifica por separado la parte algebraica y cada logaritmo, para que no se mezclen en una fracción."""
    F = sp.expand(F)
    logs = [t for t in sp.Add.make_args(F) if t.has(sp.log)]
    resto = [t for t in sp.Add.make_args(F) if not t.has(sp.log)]
    return sp.simplify(sp.Add(*resto)) + sp.Add(*[sp.simplify(t) for t in logs])


def sustitucion_trig(f):
    """(caso, a, x(θ), raíz en θ, integrando en θ, H(θ), F(x)); ValueError si no hay una raíz de esas formas
    o si el resultado no se puede regresar a x con el triángulo."""
    r = _raiz_cuadratica(f)
    if r is None:
        raise ValueError("no encontré √(a² - x²), √(a² + x²) ni √(x² - a²) (con x² de coeficiente 1 y sin término en x).")
    base, signo, c = r
    if signo == -1 and c > 0:
        a = sp.sqrt(c); caso = "a² - x²"; xt, raiz = a * _S, a * _K
    elif signo == 1 and c > 0:
        a = sp.sqrt(c); caso = "a² + x²"; xt, raiz = a * sp.tan(TH), a / _K
    elif signo == 1 and c < 0:
        a = sp.sqrt(-c); caso = "x² - a²"; xt, raiz = a / _K, a * sp.tan(TH)
    else:
        raise ValueError("la expresión bajo la raíz es negativa para todo x.")
    g = f.replace(lambda e: isinstance(e, sp.Pow) and sp.expand(e.base) == base and e.exp.is_Rational and e.exp.q == 2,
                  lambda e: raiz**(2 * e.exp))
    g = sp.simplify(g.subs(x, xt) * sp.diff(xt, TH))
    if g.has(x):
        raise ValueError("después de sustituir queda una x: revisa que la raíz sea la única parte con x² + c.")
    H = con_limite(lambda: sp.integrate(g, TH), 15)
    if H.has(sp.Integral):
        raise ValueError(f"sympy no encontró ∫ {g} dθ con funciones elementales.")
    H = _forma_estandar(H)
    razones, inversa = _razones(caso, a)
    F = H.subs(razones).subs(TH, inversa)
    F = con_valor_absoluto(_ordena(_sin_constantes(sp.simplify(F))))
    puntos = _puntos_dominio(caso, a)
    if not comprueba_en(F, f, puntos):
        raise ValueError("el resultado no se pudo regresar a x con el triángulo (o no es real en el dominio).")
    return caso, a, xt, raiz, g, H, F


def _puntos_dominio(caso, a):
    """Puntos de prueba dentro del dominio (sin x = 0, donde algunos integrandos como √(a² + x²)/x no existen)."""
    a = float(a)
    return {"a² - x²": [-0.7 * a, -0.3 * a, 0.2 * a, 0.5 * a, 0.8 * a],
            "a² + x²": [-2 * a, -0.5 * a, 0.3 * a, a, 2.5 * a],
            "x² - a²": [1.2 * a, 1.5 * a, 2 * a, 3 * a, 4 * a]}[caso]


def comprueba_en(F, f, puntos):
    """True si F es real y F' = f en todos los puntos (puntos dentro del dominio de f)."""
    dF = derivada_legible(F)
    for p in puntos:
        try:
            if abs(valor_real(F, p)) > 1e12 or abs(valor_real(dF, p) - valor_real(f, p)) > 1e-9:
                return False
        except (ValueError, TypeError, ZeroDivisionError, OverflowError):
            return False
    return True


def dibuja_triangulo(caso, a):
    lados = {"a² - x²": ("x", "√(a² - x²)", "a"), "a² + x²": ("x", "a", "√(a² + x²)"),
             "x² - a²": ("√(x² - a²)", "a", "x")}[caso]           # (opuesto, adyacente, hipotenusa)
    fig, ax = plt.subplots(figsize=(3, 2.2))
    ax.plot([0, 2, 2, 0], [0, 0, 1.3, 0], color="black", lw=1.5)
    ax.text(2.06, 0.6, lados[0]); ax.text(0.85, -0.18, lados[1]); ax.text(0.7, 0.78, lados[2])
    ax.text(0.35, 0.06, "θ"); ax.set_axis_off(); ax.set_title(f"caso {caso}, a = {a}", fontsize=9)
    plt.show()


def calculadora_sustitucion_trig(f_txt, dibujar):
    try:
        f = parsear(f_txt)
        caso, a, xt, raiz, g, H, F = sustitucion_trig(f)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print(f"caso {caso} con a = {a}:  x = {xt},  la raíz vale {raiz}" + ("   (para x > a)" if caso == "x² - a²" else ""))
    print("integrando en θ:", g)
    print("integral en θ:", H, "+ C")
    print("con el triángulo, de regreso a x:", mostrar_antiderivada(F))
    print("derivada del resultado = f(x) en el dominio: sí (comprobado en 5 puntos)")
    if dibujar:
        dibuja_triangulo(caso, a)


widgets.interact(calculadora_sustitucion_trig,
    f_txt=widgets.Text(value="x^2/sqrt(4-x^2)", description="f(x) =", continuous_update=False),
    dibujar=widgets.Checkbox(value=True, description="dibujar el triángulo"));

**Calculadora de valor eficaz.** $V_{\mathrm{rms}}=\sqrt{\frac1T\int_0^Tv^2\,dt}$. Escribe $v(t)$ y el periodo $T$ en segundos (puedes escribir `1/60` o `2*pi`). Con *media onda*, la señal vale $v(t)$ en la primera mitad del periodo y $0$ en la segunda.

In [ ]:
def valor_eficaz(v, T, media_onda=False):
    """√((1/T) ∫_0^T v² dt); con media_onda, v solo cuenta en [0, T/2]. T puede ser texto ("1/60", "2*pi")."""
    Tn = parsear(T) if isinstance(T, str) else sp.nsimplify(T, [sp.pi])
    if Tn.free_symbols or not Tn.is_real or not Tn > 0:
        raise ValueError("el periodo debe ser un número positivo.")
    fin = Tn / 2 if media_onda else Tn
    for k in (0.13, 0.37, 0.71, 0.93):                  # v debe ser real en el intervalo
        try:
            valor_real(v, k * fin)
        except ValueError:
            raise ValueError(f"v no tiene valor real en t = {cifras(float(k * fin), 3)} s: revisa la función.") from None
    I = con_limite(lambda: sp.integrate(v**2, (x, 0, fin)), 15)
    especiales = [a for a in I.atoms(sp.Function) if not isinstance(a, _PERMITIDAS)]
    if I.has(sp.Integral) or especiales:                # sin antiderivada elemental: cuadratura
        valor, err = quad(sp.lambdify(x, v**2, "math"), 0, float(fin), limit=200)
        if not math.isfinite(valor) or err > 1e-8 * max(1.0, abs(valor)):
            raise ValueError("la integral de v² no converge en el periodo: v no tiene valor eficaz finito.")
        return sp.Float(math.sqrt(valor / float(Tn)), 15)
    if not I.is_finite:
        raise ValueError("la integral de v² diverge en el periodo: v no tiene valor eficaz finito.")
    return sp.sqrt(I / Tn)


def calculadora_eficaz(v_txt, T_txt, media_onda, n_cifras):
    try:
        v = parsear(v_txt)
        r = valor_eficaz(v, T_txt, media_onda)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    exacto = "" if isinstance(r, sp.Float) else f" = {sp.nsimplify(sp.simplify(r))}"
    print(f"V_rms{exacto} ≈ {cifras(r, n_cifras)}   ({n_cifras} cifras significativas, redondeado; ángulos en radianes)")
    if isinstance(r, sp.Float):
        print("(v² no tiene antiderivada elemental: el valor se obtuvo con una suma numérica)")


widgets.interact(calculadora_eficaz,
    v_txt=widgets.Text(value="179.6*sin(120*pi*t)", description="v(t) =", continuous_update=False),
    T_txt=widgets.Text(value="1/60", description="T (s) =", continuous_update=False),
    media_onda=widgets.Checkbox(value=False, description="media onda"),
    n_cifras=widgets.IntSlider(value=4, min=1, max=8, description="cifras sig.", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 4 (resultado conocido)
PRUEBAS_4 = [
    ("sen³: caso (a), -cos x + cos³x/3", lambda: (lambda r: r[0].startswith("(a)") and
     sp.simplify(r[2] - (-sp.cos(x) + sp.cos(x)**3 / 3)) == 0)(potencias_trig(3, 0))),
    ("sen² cos³: caso (b), F' = f", lambda: (lambda r: r[0].startswith("(b)") and
     compara_con_sympy(r[2], sp.sin(x)**2 * sp.cos(x)**3)[0])(potencias_trig(2, 3))),
    ("sen² cos²: caso (c), 1/8 - cos 4x/8", lambda: (lambda r: r[0].startswith("(c)") and
     sp.simplify(r[1] - (sp.Rational(1, 8) - sp.cos(4 * x) / 8)) == 0)(potencias_trig(2, 2))),
    ("√(4 - x²): caso a² - x², x√(4 - x²)/2 + 2 arcsen(x/2)", lambda: (lambda r: r[0] == "a² - x²" and
     misma_funcion(r[6], x * sp.sqrt(4 - x**2) / 2 + 2 * sp.asin(x / 2)))(sustitucion_trig(sp.sqrt(4 - x**2)))),
    ("(x² + 9)^(-3/2): caso a² + x², x/(9√(x² + 9))", lambda: (lambda r: r[0] == "a² + x²" and
     misma_funcion(r[6], x / (9 * sp.sqrt(x**2 + 9))))(sustitucion_trig((x**2 + 9)**sp.Rational(-3, 2)))),
    ("1/√(x² + 4): resultado real, ln|x + √(x² + 4)|", lambda: (lambda r: not r[6].has(sp.I) and
     comprueba_en(r[6], 1 / sp.sqrt(x**2 + 4), [-3, -0.5, 0.4, 2, 7]))(sustitucion_trig(1 / sp.sqrt(x**2 + 4)))),
    ("valor eficaz de sen(120πt) con T = 1/60: 1/√2", lambda: valor_eficaz(sp.sin(120 * sp.pi * x), "1/60") == 1 / sp.sqrt(2)),
    ("valor eficaz de la rampa v = t con T = 1: 1/√3", lambda: valor_eficaz(x, "1") == 1 / sp.sqrt(3)),
    ("valor eficaz de v = 1 con media onda: 1/√2", lambda: valor_eficaz(sp.Integer(1), "2", True) == 1 / sp.sqrt(2)),
]
for nombre, prueba in PRUEBAS_4:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 4).** Calcula a mano $\int_0^{\pi}\sin^2x\,dx$ y escribe el número.

In [ ]:
mi_valor = None     # escribe un número (también sirve "pi/2")

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    ref = sp.integrate(sp.sin(x)**2, (x, 0, sp.pi))
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        print("La calculadora da:", ref, "≈", cifras(ref, 6))
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: sen² x = (1 - cos 2x)/2, y cos 2x acumula cero en [0, π]")

## 5. Integrales impropias

$$\int_a^{\infty}f(x)\,dx=\lim_{R\to\infty}\int_a^Rf(x)\,dx,\qquad\int_a^bf(x)\,dx=\lim_{\varepsilon\to0^+}\int_{a+\varepsilon}^bf(x)\,dx\ \ (f\text{ no acotada cerca de }a).$$

Si hay un punto problemático dentro del intervalo, o los dos extremos son infinitos, la integral se separa, y **cada parte debe converger por su lado**. Integrales $p$: $\int_1^{\infty}x^{-p}dx$ converge si $p>1$; $\int_0^1x^{-p}dx$ converge si $p<1$.

In [ ]:
print(" R       ∫_1^R dx/x²   ∫_1^R dx/x")
for R in (10, 100, 1000, 10000):
    print(f"{R:<7d} {cifras(1 - 1 / R, 6):>12} {cifras(math.log(R), 6):>12}")

**Explorador de integrales impropias.** Escribe $f(x)$ y los extremos (pueden ser `-inf`, `inf`, `pi/2` o una fracción). El explorador busca los puntos de $[a,b]$ donde $f$ no está definida, separa la integral, arma la tabla de integrales parciales de cada parte y da un veredicto por parte y para la integral completa. El veredicto sale de un cálculo exacto (`sympy.integrate` o el límite de la antiderivada); si solo hay evidencia numérica, el explorador lo dice y no afirma que la integral converja o diverja. Si una parte diverge, no suma las demás.

In [ ]:
def lee_extremo(texto):
    """'inf', '-inf', 'pi/2', '3', '0.5' -> número de sympy (exacto si se puede)."""
    t = str(texto).strip().lower().replace(" ", "").replace("∞", "inf")
    if t in ("inf", "+inf", "oo", "+oo", "infinito"):
        return sp.oo
    if t in ("-inf", "-oo", "-infinito"):
        return -sp.oo
    try:
        v = parsear(t)
    except ValueError as err:
        raise ValueError("Escribe los extremos como números (3, 0.5, 1/2, pi/2), inf o -inf.") from err
    if v.free_symbols:
        raise ValueError("Los extremos son números: no lleves x en ellos.")
    return v


def _txt(v):
    """Extremo legible: ∞, un entero o fracción tal cual, o el valor exacto con su aproximación."""
    if v in (sp.oo, -sp.oo):
        return "∞" if v == sp.oo else "-∞"
    return str(v) if v.is_Rational else f"{v} ≈ {cifras(v, 6)}"


def _problemas(f, a, b):
    """Puntos de [a, b] donde f no es continua; ValueError si no son un número finito de puntos."""
    try:
        dom = con_limite(lambda: sp.calculus.util.continuous_domain(f, x, sp.Interval(a, b)), 10)
    except (ValueError, NotImplementedError):
        raise ValueError("no pude determinar dónde f no está definida; prueba con un intervalo más corto.") from None
    fuera = sp.Interval(a, b) - dom
    if fuera == sp.EmptySet:
        return []
    if isinstance(fuera, sp.FiniteSet):
        return sorted(fuera, key=float)
    if isinstance(fuera, sp.ConditionSet) or fuera.has(sp.ConditionSet):
        raise ValueError("no pude determinar dónde f no está definida; prueba con un intervalo más corto.")
    if fuera.is_iterable or fuera.has(sp.ImageSet):
        raise ValueError("f tiene una infinidad de puntos problemáticos en el intervalo (como tan x en [0, ∞)): "
                         "el explorador no puede separar la integral en un número finito de partes.")
    raise ValueError("f no está definida (o no es real) en todo un tramo del intervalo.")


def _revisa_real(f, p, q):
    """ValueError si f da valores complejos dentro de la parte (p, q) (por ejemplo x^(-1/3) con x < 0)."""
    lo = float(p) if p != -sp.oo else min(float(q) - 10, -10.0)
    hi = float(q) if q != sp.oo else max(float(p) + 10, 10.0)
    for k in (0.17, 0.41, 0.63, 0.88):
        try:
            valor_real(f, sp.nsimplify(lo + k * (hi - lo)))
        except ValueError:
            raise ValueError(f"f no es real dentro de [{_txt(p)}, {_txt(q)}]. Si escribiste una raíz cúbica de un "
                             "número negativo, sympy la toma compleja; usa un intervalo donde la base sea positiva.") from None


def _partes(a, b, malos):
    """Separa [a, b] para que cada parte tenga a lo más un extremo problemático."""
    cortes = sorted({a, b, *malos}, key=lambda v: float(v))
    partes = []
    for p, q in zip(cortes, cortes[1:]):
        mp, mq = (p == -sp.oo or p in malos), (q == sp.oo or q in malos)
        if mp and mq:
            if p == -sp.oo and q == sp.oo:
                c = sp.Integer(0)
            elif q == sp.oo:
                c = p + 1
            elif p == -sp.oo:
                c = q - 1
            else:
                c = (p + q) / 2
            partes += [(p, c, "izq"), (c, q, "der")]
        else:
            partes.append((p, q, "izq" if mp else ("der" if mq else None)))
    return partes


def _quad(fn, u, v):
    """∫_u^v por cuadratura, en tramos de longitud <= 50 (las oscilaciones largas no confunden a quad)."""
    cortes = list(np.linspace(u, v, max(2, int(math.ceil(abs(v - u) / 50)) + 1)))
    total = 0.0
    with np.errstate(all="ignore"):
        for s_, t_ in zip(cortes, cortes[1:]):
            total += quad(fn, s_, t_, limit=200)[0]
    return total


def _acercamiento(p, q, lado):
    """Los cuatro puntos z de la tabla, cada vez más cerca del extremo problemático."""
    largo = 1.0 if (p == -sp.oo or q == sp.oo) else float(q - p)
    eps = [10.0**-k if largo >= 1 else largo / 10**k for k in (1, 2, 3, 4)]
    if lado == "der":
        return [(10**k if 10**k > p else p + 10**k) for k in (1, 2, 3, 4)] if q == sp.oo else [float(q) - e for e in eps]
    return [(-10**k if -10**k < q else q - 10**k) for k in (1, 2, 3, 4)] if p == -sp.oo else [float(p) + e for e in eps]


def _parciales(F, f, p, q, lado):
    """Tabla de integrales parciales hacia el extremo problemático. Con antiderivada elemental, F(z) - F(p);
    sin ella, sumas de cuadraturas que se acumulan tramo por tramo desde el extremo bueno."""
    zs, filas = _acercamiento(p, q, lado), []
    if F is not None:
        for z in zs:
            try:
                v = valor_real(F, z) - valor_real(F, p) if lado == "der" else valor_real(F, q) - valor_real(F, z)
            except OverflowError:
                v = math.inf
            except (ValueError, TypeError, ZeroDivisionError):
                v = float("nan")
            filas.append((z, v))
        return filas
    fn = sp.lambdify(x, f, "math")
    bueno = float(p) if lado == "der" else float(q)
    acumulado, previo = 0.0, bueno
    for z in zs:
        try:
            tramo = _quad(fn, previo, float(z))
            acumulado += tramo if lado == "der" else -tramo
        except OverflowError:
            acumulado = math.inf
        except (ValueError, ZeroDivisionError):
            acumulado = float("nan")
        previo = float(z)
        filas.append((z, acumulado))
    return filas


def _veredicto(I):
    """(valor, estado) a partir de un resultado exacto de sympy, o None si no sirve para decidir."""
    if I is None or I.has(sp.Integral, sp.nan) or I is sp.nan:
        return None
    if isinstance(I, sp.AccumBounds) or I.has(sp.AccumBounds):
        return None, "oscila"
    if I.has(sp.oo, -sp.oo, sp.zoo) or I.is_finite is False:
        return None, "diverge"
    if I.is_real is False or I.has(sp.I):
        return None
    return sp.simplify(I), "converge"


def _valor_parte(f, F, p, q, lado):
    """(valor, estado) de una parte: estado es converge, diverge, oscila o no concluyente."""
    try:
        r = _veredicto(con_limite(lambda: sp.integrate(f, (x, p, q)), 15))
    except (ValueError, NotImplementedError):
        r = None
    if r is None and F is not None and lado is not None:
        try:
            if lado == "der":
                lim = con_limite(lambda: sp.limit(F, x, q, "-"), 10)
                r = _veredicto(lim - F.subs(x, p))
            else:
                lim = con_limite(lambda: sp.limit(F, x, p, "+"), 10)
                r = _veredicto(F.subs(x, q) - lim)
        except (ValueError, NotImplementedError):
            r = None
    return r if r is not None else (None, "no concluyente")


def explorar(f, a, b):
    """Lista de partes (p, q, tabla, valor, estado) y el veredicto global (valor, estado)."""
    if not (a < b):
        raise ValueError("el extremo izquierdo debe ser menor que el derecho.")
    malos = _problemas(f, a, b)
    try:
        F = antiderivada(f)
    except ValueError:
        F = None
    resultado = []
    for p, q, lado in _partes(a, b, malos):
        _revisa_real(f, p, q)
        valor, estado = _valor_parte(f, F, p, q, lado)
        tabla = _parciales(F, f, p, q, lado) if lado is not None else []
        resultado.append((p, q, tabla, valor, estado))
    estados = [r[4] for r in resultado]
    if "diverge" in estados or "oscila" in estados:
        return resultado, (None, "diverge")
    if "no concluyente" in estados:
        return resultado, (None, "no concluyente")
    return resultado, (sp.simplify(sum(r[3] for r in resultado)), "converge")


_LEYENDA = {"diverge": "DIVERGE", "oscila": "DIVERGE: las integrales parciales oscilan sin acercarse a un número",
            "no concluyente": "NO CONCLUYENTE: sympy no dio un valor exacto; mira la tabla, pero una tabla sola "
                              "no demuestra que la integral converja o diverja"}


def explorador_impropias(f_txt, a_txt, b_txt):
    try:
        f = parsear(f_txt)
        a, b = lee_extremo(a_txt), lee_extremo(b_txt)
        partes, (total, estado) = explorar(f, a, b)
    except ValueError as err:
        print("Revisa la entrada:", err); return
    print("(ángulos en radianes; parciales con 6 cifras significativas, redondeadas)")
    for p, q, tabla, valor, est in partes:
        print(f"parte {'(' if p == -sp.oo else '['}{_txt(p)}, {_txt(q)}{')' if q == sp.oo else ']'}:")
        for z, v in tabla:
            print(f"   hasta {cifras(z, 6):>10}:  {'más de 1e308' if math.isinf(v) else cifras(v, 6)}")
        print("   →", f"converge a {valor} ≈ {cifras(valor, 6)}" if est == "converge" else _LEYENDA[est])
    if estado == "diverge":
        print("La integral completa DIVERGE: al menos una parte diverge, y las partes divergentes no se suman.")
    elif estado == "no concluyente":
        print("La integral completa queda SIN VEREDICTO: al menos una parte solo tiene evidencia numérica.")
    else:
        print(f"La integral completa converge a {total} ≈ {cifras(total, 6)}")


widgets.interact(explorador_impropias,
    f_txt=widgets.Text(value="1/x^2", description="f(x) =", continuous_update=False),
    a_txt=widgets.Text(value="1", description="a =", continuous_update=False),
    b_txt=widgets.Text(value="inf", description="b =", continuous_update=False));

In [ ]:
# Casos de prueba de la sección 5 (los del prompt del libro y cuatro más)
def _valor(f, a, b):
    return explorar(f, a, b)[1][0]


def _estado(f, a, b):
    return explorar(f, a, b)[1][1]


PRUEBAS_5 = [
    ("e^(-2x) en [0, ∞): 1/2", lambda: _valor(sp.exp(-2 * x), 0, sp.oo) == sp.Rational(1, 2)),
    ("1/x en [1, ∞): diverge; parciales 2.30259, 4.60517, 6.90776, 9.21034",
     lambda: (lambda r: r[1][1] == "diverge" and [cifras(v, 6) for _, v in r[0][0][2]] == ["2.30259", "4.60517", "6.90776", "9.21034"])(
         explorar(1 / x, 1, sp.oo))),
    ("1/√x en [0, 1]: 2", lambda: _valor(1 / sp.sqrt(x), 0, 1) == 2),
    ("1/x en [-1, 1]: diverge aunque sea impar", lambda: _estado(1 / x, -1, 1) == "diverge"),
    ("1/(1 + x²) en (-∞, ∞): π", lambda: _valor(1 / (1 + x**2), -sp.oo, sp.oo) == sp.pi),
    ("x^(-1/3) en [0, 1]: 3/2", lambda: _valor(x**sp.Rational(-1, 3), 0, 1) == sp.Rational(3, 2)),
    ("sen x / x en [0, ∞): π/2 (sin antiderivada elemental)", lambda: sp.simplify(_valor(sp.sin(x) / x, 0, sp.oo) - sp.pi / 2) == 0),
    ("sen x en [0, ∞): diverge porque oscila", lambda: (lambda r: r[0][0][4] == "oscila" and r[1][1] == "diverge")(
        explorar(sp.sin(x), 0, sp.oo))),
]
for nombre, prueba in PRUEBAS_5:
    print("OK  " if prueba() else "FALLA", nombre)

**Contrasta (sección 5).** Calcula a mano $\int_1^{\infty}\frac{dx}{x^3}$ y escribe el número.

In [ ]:
mi_valor = None     # escribe un número (también sirve "1/2")

if mi_valor is None:     # la referencia se muestra después de tu cálculo
    print("falta tu cálculo: escríbelo arriba y vuelve a ejecutar la celda")
else:
    try:
        mio = a_numero(mi_valor)
    except ValueError as err:
        print("Revisa tu número:", err)
    else:
        ref = _valor(1 / x**3, 1, sp.oo)
        print("El explorador da:", ref)
        print("coinciden a 3 cifras" if coincide(mio, ref) else
              "NO coinciden: ∫_1^R x^(-3) dx = 1/2 - 1/(2R²); toma el límite")

## Verificación final
Ejecuta esta celda al terminar. Debe imprimir que todas las pruebas pasaron.

In [ ]:
todas = PRUEBAS_1 + PRUEBAS_2 + PRUEBAS_3 + PRUEBAS_4 + PRUEBAS_5
fallas = [nombre for nombre, prueba in todas if not prueba()]
assert not fallas, f"Fallan {len(fallas)} pruebas: {fallas}"
print(f"Todas las verificaciones pasaron: {len(todas)}/{len(todas)}")